# Phase 3 · Model Evaluation (Custom CNN)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MartinAlani/project_deep_learning_week_7/blob/main/notebooks/03_evaluation.ipynb)

**Goals of this notebook**
1. Load the custom CNN trained in Phase 2 (no retraining needed).
2. Evaluate it on the **validation** set and, for the first time, on the untouched **test** set.
3. Compute **accuracy, precision, recall and F1-score**, overall and per class.
4. Visualize the **confusion matrix** to see which classes the model confuses.
5. Look at misclassified images to understand the errors.

> 💡 No training happens here, so a GPU is optional; the notebook runs in a few minutes.

## Metrics cheat sheet

| Metric | Question it answers | Formula |
|---|---|---|
| **Accuracy** | What fraction of all images is correct? | correct / total |
| **Precision** | When the model predicts "cat", how often is it a cat? | TP / (TP + FP) |
| **Recall** | Of all real cats, how many did the model find? | TP / (TP + FN) |
| **F1-score** | Balance between precision and recall | 2·P·R / (P + R) |

TP = true positives, FP = false positives, FN = false negatives (computed per class).

## 1. Setup: imports and Google Drive

We mount Google Drive to load the model saved in Phase 2 and to save the evaluation results.

In [ ]:
import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)

SEED = 42
keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/project_deep_learning_week_7"
except ImportError:
    PROJECT_DIR = ".."

MODELS_DIR = os.path.join(PROJECT_DIR, "models")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Models folder: ", MODELS_DIR)
print("Results folder:", RESULTS_DIR)

## 2. Data (same split as before)

Same `SEED` → exactly the same validation set as in Phase 2, plus the **test set**, which we use now for the first time. We do **not** shuffle these sets, so predictions stay aligned with the true labels.

In [ ]:
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]
BATCH_SIZE = 64

(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train_full, y_test = y_train_full.flatten(), y_test.flatten()

_, x_val, _, y_val = train_test_split(
    x_train_full, y_train_full, test_size=5000, stratify=y_train_full, random_state=SEED
)

def make_dataset(x, y):
    """Batched, prefetched tf.data.Dataset without shuffling (keeps label order)."""
    return tf.data.Dataset.from_tensor_slices((x, y)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = make_dataset(x_val, y_val)
test_ds = make_dataset(x_test, y_test)

print("Validation:", x_val.shape, "| Test:", x_test.shape)

## 3. Load the trained model

`load_model` restores the full model: architecture, trained weights and preprocessing layers. Data augmentation layers are automatically inactive at inference time.

In [ ]:
MODEL_PATH = os.path.join(MODELS_DIR, "custom_cnn.keras")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model not found at {MODEL_PATH}. Run notebook 02 first (it saves the model to Drive)."
    )

model = keras.models.load_model(MODEL_PATH)
print(f"Loaded '{model.name}' with {model.count_params():,} parameters")

## 4. Overall accuracy: validation vs test

If both numbers are close, the validation set was a good proxy and our training decisions did not overfit to it.

In [ ]:
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
test_loss, test_acc = model.evaluate(test_ds, verbose=0)

summary = pd.DataFrame(
    {"loss": [val_loss, test_loss], "accuracy": [val_acc, test_acc]},
    index=["validation", "test"],
).round(4)
summary

## 5. Predictions on the test set

`model.predict` returns, for every image, the **10 softmax probabilities**. The predicted class is the one with the highest probability (`argmax`).

In [ ]:
test_probs = model.predict(test_ds, verbose=0)    # shape (10000, 10)
y_pred = np.argmax(test_probs, axis=1)             # predicted class per image
confidence = np.max(test_probs, axis=1)            # probability of the predicted class

print("Probabilities shape:", test_probs.shape)
print("Example — probabilities for the first test image:")
for name, p in zip(CLASS_NAMES, test_probs[0]):
    print(f"  {name:>11}: {p:6.2%}")
print(f"True class: {CLASS_NAMES[y_test[0]]} | Predicted: {CLASS_NAMES[y_pred[0]]}")
print(f"\nTest accuracy (from predictions): {accuracy_score(y_test, y_pred):.4f}")

## 6. Precision, recall and F1-score per class

`classification_report` computes the metrics for each class:
- **support** = number of real images of that class (1,000 each in the test set)
- **macro avg** = simple average over the 10 classes (every class counts equally)
- **weighted avg** = average weighted by support (identical to macro here, because the test set is balanced)

In [ ]:
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0))

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred, zero_division=0)

per_class = pd.DataFrame(
    {"precision": precision, "recall": recall, "f1_score": f1, "support": support},
    index=CLASS_NAMES,
).sort_values("f1_score", ascending=False)
per_class.round(4)

A bar chart makes it easy to compare classes at a glance:

In [ ]:
ax = per_class[["precision", "recall", "f1_score"]].plot(kind="bar", figsize=(12, 5), width=0.8)
ax.set_title("Custom CNN · precision, recall and F1-score per class (test set)")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.axhline(test_acc, color="gray", linestyle="--", label=f"overall accuracy ({test_acc:.2%})")
ax.legend(loc="lower left")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "custom_cnn_metrics_per_class.png"), dpi=120)
plt.show()

## 7. Confusion matrix

- **Rows** = true class, **columns** = predicted class.
- The **diagonal** holds the correct predictions; everything off the diagonal is an error.
- Left: raw counts. Right: normalized by row, so each row shows the percentage of that class that went to each prediction (the diagonal is the per-class **recall**).

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False)
axes[0].set_title("Confusion matrix · counts (test set)")
sns.heatmap(cm_norm, annot=True, fmt=".0%", cmap="Blues", ax=axes[1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False)
axes[1].set_title("Confusion matrix · normalized by true class")
for ax in axes:
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "custom_cnn_confusion_matrix.png"), dpi=120)
plt.show()

## 8. Most frequent confusions

We list the largest off-diagonal cells: the pairs of classes the model mixes up most often.

In [ ]:
errors = cm.copy()
np.fill_diagonal(errors, 0)   # ignore correct predictions

top_pairs = [
    (CLASS_NAMES[i], CLASS_NAMES[j], errors[i, j])
    for i, j in zip(*np.unravel_index(np.argsort(errors, axis=None)[::-1][:10], errors.shape))
]
top_confusions = pd.DataFrame(top_pairs, columns=["true class", "predicted as", "count"])
top_confusions

## 9. What do the mistakes look like?

We show misclassified test images with the true label, the predicted label and the model's confidence. This helps explain *why* the model fails (blurry images, unusual angles, similar classes...).

In [ ]:
def show_mistakes(n=24, most_confident=True):
    """Display misclassified test images.

    Args:
        n: number of images to show.
        most_confident: if True, show the errors the model was MOST sure about;
            otherwise show random errors.
    """
    wrong = np.where(y_pred != y_test)[0]
    if most_confident:
        wrong = wrong[np.argsort(confidence[wrong])[::-1]]
    else:
        wrong = np.random.choice(wrong, size=min(n, len(wrong)), replace=False)
    wrong = wrong[:n]

    n_cols = 8
    n_rows = int(np.ceil(len(wrong) / n_cols))
    plt.figure(figsize=(n_cols * 2, n_rows * 2.4))
    for k, i in enumerate(wrong):
        plt.subplot(n_rows, n_cols, k + 1)
        plt.imshow(x_test[i])
        plt.title(f"true: {CLASS_NAMES[y_test[i]]}\npred: {CLASS_NAMES[y_pred[i]]} ({confidence[i]:.0%})",
                  fontsize=8, color="crimson")
        plt.axis("off")
    title = "Most confident mistakes" if most_confident else "Random mistakes"
    plt.suptitle(f"{title} (test set)", fontsize=14)
    plt.tight_layout()
    plt.show()

print(f"Misclassified test images: {(y_pred != y_test).sum()} of {len(y_test)}")
show_mistakes(most_confident=True)

In [ ]:
show_mistakes(most_confident=False)

## 10. Save the evaluation results

We save every metric to a JSON file. In Phase 4 we will load it to compare the custom CNN with the transfer-learning models.

In [ ]:
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro", zero_division=0
)

evaluation = {
    "model": "custom_cnn",
    "val_accuracy": float(val_acc),
    "val_loss": float(val_loss),
    "test_accuracy": float(test_acc),
    "test_loss": float(test_loss),
    "test_macro_precision": float(macro_p),
    "test_macro_recall": float(macro_r),
    "test_macro_f1": float(macro_f1),
    "per_class": per_class.round(4).to_dict(orient="index"),
    "confusion_matrix": cm.tolist(),
    "top_confusions": top_confusions.to_dict(orient="records"),
}

with open(os.path.join(RESULTS_DIR, "custom_cnn_evaluation.json"), "w") as f:
    json.dump(evaluation, f, indent=2, default=int)

print(f"Test accuracy:   {test_acc:.4f}")
print(f"Macro precision: {macro_p:.4f}")
print(f"Macro recall:    {macro_r:.4f}")
print(f"Macro F1-score:  {macro_f1:.4f}")
print("\nSaved files in", RESULTS_DIR)
for name in sorted(os.listdir(RESULTS_DIR)):
    print(" -", name)

## ✅ Phase 3 summary

- The model is evaluated on the **test set** (10,000 images never used for training or decisions) → unbiased estimate of real-world performance.
- Per-class **precision / recall / F1** show which classes are easy and which are hard.
- The **confusion matrix** reveals *which* classes are mixed up, typically visually similar ones (e.g. cat ↔ dog, automobile ↔ truck).

👉 **For the report:** save the per-class bar chart, the confusion matrix and the grid of mistakes, and note the test accuracy and macro F1-score.

**Next step → Phase 4:** transfer learning with pre-trained models, and comparison with this custom CNN.